In [1]:
# whisperx and openai-whisper (MUST)
!pip install whisperx openai-whisper
print("✅ whisperx and openai-whisper installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:0

✅ whisperx and openai-whisper installed!


In [1]:
# ════════════════════════════════════════════════════════════════════
# Settings — edit before running.
# ════════════════════════════════════════════════════════════════════
SOURCE = "drive"           # "zip" (upload plan + review/) or "drive"
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/_Phase3/sources"
ARCHIVE_ZIP_NAME = "Archive.zip"

# ── Book Title & Main Character  ────────────────────────────────────
BOOK_TITLE     = "مذكرات جعفر العسكري"
CHARACTER_NAME = "Jafar al-Askari"

# ── Look ────────────────────────────────────────────────────────────
BOOK_COVER_PICK   = 1          # 1..N over resources/book_cover/
BOOK_COVER_FIT    = "contain"  # fill | contain | blur_pad
BOOK_COVER_ALIGN  = "right"    # center | left | right
TYPOGRAPHY_FAMILY = "B"        # A | B | C
GRADE             = "warm"  # warm | cool | neutral | bw
CAPTION_BACKPLATE = "off"      # off | subtle | solid
TEXT_SCRIM        = "off"      # off | soft | band  (typography-over-image plate)
MUSIC_DB          = -13.0      # music bed level in dB (default -18)

# ── Text styling (blank = pipeline default) ─────────────────────────
TITLE_SIZE    = 1.0   # main-title size multiplier (1.2 = 20% larger)
TITLE_COLOR   = ""    # "#RRGGBB" or "" -> family default (aged gold)
CAPTION_SIZE  = 1.0   # caption size multiplier
CAPTION_COLOR = ""    # "#RRGGBB" or "" -> white
CAPTION_POS   = ""    # fraction of height from bottom, e.g. "0.08"; "" -> default
# NOTE: captions are disabled below via --no-captions; the CAPTION_* knobs
# take effect only if you remove that flag in the render cell.

# ── Output Files & Directories ──────────────────────────────────────
OUTPUT_BASE_DIR         = "output"
OUTPUT_FILE             = f"{OUTPUT_BASE_DIR}/final_cut_{TYPOGRAPHY_FAMILY}.mp4"
LOG_FILE                = f"{OUTPUT_BASE_DIR}/render.log"
CONDITIONED_ZIP_FILE    = f"{OUTPUT_BASE_DIR}/conditioned.zip"
FINAL_ZIP_FILE          = "output_files.zip"
RO_ZIP_FILE             = "output_files_ro.zip"
DRIVE_SAVE_DIR          = "/content/drive/MyDrive/_Phase3/output"
DRIVE_SAVE_RO_DIR       = "/content/drive/MyDrive/_Phase3/output/ro"

# Generated artifacts — supplied at render time (upload .zip or Drive):
PLAN_FILE   = f"{OUTPUT_BASE_DIR}/re_generated_plan.json"
REVIEW_DIR  = f"{OUTPUT_BASE_DIR}/review"
REVIEW_FILE = f"{OUTPUT_BASE_DIR}/review.zip"

# Committed inputs
SCRIPT_FILE = "resources/script/main_script.txt"
AUDIO_FILE  = "resources/audio/narration.mp3"
MUSIC_BED   = "resources/audio/bg_music.mp3"

# print(f"SOURCE={SOURCE!r}  OUTPUT={OUTPUT_FILE!r}  GRADE={GRADE!r}  SCRIM={TEXT_SCRIM!r}")
print(f"OUTPUT= {OUTPUT_FILE}\nGRADE = {GRADE}\nSCRIM = {TEXT_SCRIM}\n")

OUTPUT= output/final_cut_B.mp4
GRADE = warm
SCRIM = off



In [2]:
# Script source: GitHub repo
import os
import shutil

# github_url_path = "https://github.com/abdoljh/Lamahat/tree/main/_Phase3"
github_url_path = "https://github.com/abdoljh/Assemble-Video"

# Extract the base GitHub repository URL and the subfolder path
def parse_github_path(url):
    parts = url.split('/tree/main/')
    repo_url = parts[0]
    subfolder_path = parts[1] if len(parts) > 1 else ''
    # Add .git for cloning
    repo_url_for_clone = repo_url + '.git'
    return repo_url_for_clone, subfolder_path

repo_url_for_clone, subfolder_path_in_repo = parse_github_path(github_url_path)

repo_name = repo_url_for_clone.split('/')[-1].replace('.git', '')
temp_clone_dir = os.path.join('/tmp', repo_name)
dest_dir_colab = '/content'

print(f"Cloning repository: {repo_url_for_clone}")
print(f"Target subfolder in repo: {subfolder_path_in_repo}")

# Clean up any previous clone to avoid issues
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir)
    print(f"Removed existing temporary directory: {temp_clone_dir}")

# Clone the repository
clone_command = f"git clone {repo_url_for_clone} {temp_clone_dir}"
print(f"Executing: {clone_command}")
os.system(clone_command)

# Check if cloning was successful
if not os.path.exists(temp_clone_dir):
    print(f"Error: Failed to clone repository {repo_url_for_clone}")
else:
    source_dir_to_copy = os.path.join(temp_clone_dir, subfolder_path_in_repo)
    if not os.path.exists(source_dir_to_copy):
        print(f"Error: Subfolder '{subfolder_path_in_repo}' not found in cloned repository at '{source_dir_to_copy}'")
    else:
        print(f"Source directory to copy: {source_dir_to_copy}")
        print(f"Copying contents of '{source_dir_to_copy}' directly into '{dest_dir_colab}'.")

        # Define directories to skip
        dirs_to_skip = ['artifacts', 'review']

        try:
            for item in os.listdir(source_dir_to_copy):
                if item in dirs_to_skip:
                    print(f"Skipping directory '{item}' as requested.")
                    continue

                source_item_path = os.path.join(source_dir_to_copy, item)
                dest_item_path = os.path.join(dest_dir_colab, item)

                # If item already exists in destination, remove it to avoid errors
                if os.path.exists(dest_item_path):
                    if os.path.isdir(dest_item_path):
                        shutil.rmtree(dest_item_path)
                        print(f"Removed existing directory '{dest_item_path}'.")
                    else:
                        os.remove(dest_item_path)
                        print(f"Removed existing file '{dest_item_path}'.")

                if os.path.isdir(source_item_path):
                    shutil.copytree(source_item_path, dest_item_path)
                else:
                    shutil.copy2(source_item_path, dest_item_path)
                print(f"Copied '{source_item_path}' to '{dest_item_path}'.")
            print(f"Successfully copied contents of '{source_dir_to_copy}' to '{dest_dir_colab}'.")

        except Exception as e:
            print(f"An error occurred during directory contents copy: {e}")

# Clean up the cloned repository
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir)
    print(f"Cleaned up temporary clone directory: {temp_clone_dir}")

print("😺 GitHub repo loaded!")

Cloning repository: https://github.com/abdoljh/Assemble-Video.git
Target subfolder in repo: 
Executing: git clone https://github.com/abdoljh/Assemble-Video.git /tmp/Assemble-Video
Source directory to copy: /tmp/Assemble-Video/
Copying contents of '/tmp/Assemble-Video/' directly into '/content'.
Copied '/tmp/Assemble-Video/condition_assets.py' to '/content/condition_assets.py'.
Copied '/tmp/Assemble-Video/verify_title_card.py' to '/content/verify_title_card.py'.
Copied '/tmp/Assemble-Video/phase3_run.py' to '/content/phase3_run.py'.
Copied '/tmp/Assemble-Video/trim_book_cover.py' to '/content/trim_book_cover.py'.
Copied '/tmp/Assemble-Video/verify_font_discovery.py' to '/content/verify_font_discovery.py'.
Copied '/tmp/Assemble-Video/render_plan.py' to '/content/render_plan.py'.
Copied '/tmp/Assemble-Video/sandbox_test.py' to '/content/sandbox_test.py'.
Copied '/tmp/Assemble-Video/CLAUDE.md' to '/content/CLAUDE.md'.
Copied '/tmp/Assemble-Video/output' to '/content/output'.
Copied '/tmp/A

In [3]:
# Install anthropic
!pip install anthropic --quiet
print("✴️ Anthropic installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 22.0 MB/s eta 0:00:00
✴️ Anthropic installed!


In [4]:
# Colab API Keys
print("Retrieving the Anthropic, Pexels & Hugging Face API keys & token ...")
from google.colab import userdata
import os

# Retrieve the Anthropic API key from Colab Secrets
anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
pexels_api_key = userdata.get('PEXELS_API_KEY')
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable for phase3_run.py to use
if anthropic_api_key:
    os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key
    print("🔑 Anthropic API key loaded.")
else:
    print("❌ Warning: ANTHROPIC_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if pexels_api_key:
    os.environ['PEXELS_API_KEY'] = pexels_api_key
    print("🔑 Pexels API key loaded.")
else:
    print("❌ Warning: PEXELS_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("🔑 Hugging Face token loaded.")
else:
    print("❌ Warning: HF_TOKEN not found in Colab Secrets. Please ensure it's set correctly.")

Retrieving the Anthropic, Pexels & Hugging Face API keys & token ...
🔑 Anthropic API key loaded.
🔑 Pexels API key loaded.
🔑 Hugging Face token loaded.


In [5]:
# Confirm alignment works (interpolation backend — no install needed)
!python phase3_run.py \
    --script {SCRIPT_FILE} \
    --audio  {AUDIO_FILE} \
    --align-only \
    --align-backend interpolated

print("✅ Confirming alignment completed!")


Script : resources/script/main_script.txt  (3,955 chars)
INFO  phase3.typography_common  Amiri fonts loaded via repo fonts/ (next to phase3): /content/fonts
Traceback (most recent call last):
  File "/content/phase3_run.py", line 451, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/phase3_run.py", line 366, in main
    _run_align_only(args, script_text)
  File "/content/phase3_run.py", line 236, in _run_align_only
    from phase3.align import align, tokenize_script
  File "/content/phase3/__init__.py", line 56, in <module>
    from .sources import Fetcher, FetcherConfig
ImportError: cannot import name 'Fetcher' from 'phase3.sources' (/content/phase3/sources/__init__.py)
✅ Confirming alignment completed!


In [ ]:
# Regenerate the plan with the fixes
# Produces output/re_generated_plan.json
!python phase3_run.py \
    --script         {SCRIPT_FILE} \
    --audio          {AUDIO_FILE} \
    --book-title     "{BOOK_TITLE}" \
    --character-name "{CHARACTER_NAME}" \
    --plan-only \
    --save-plan      {PLAN_FILE}

print("✅ Regenerating the plan completed!")

In [ ]:
# Audit the regenerated plan (OPTIONAL)
!python audit_plan.py {PLAN_FILE}
# Expect: ~43 shots, <10% auto-split

print("✅ Audit completed!")

In [ ]:
import os
from pathlib import Path

# Ensure the parent directory for OUTPUT_FILE exists
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Construct the command using an f-string for proper variable interpolation
# Note: $ANTHROPIC_API_KEY and $PEXELS_API_KEY are environment variables,
# and are correctly handled by the shell command, so they don't need Python interpolation.
command = f"""
python prebuild_assets.py \
    --plan           \"{PLAN_FILE}\" \
    --script         \"{SCRIPT_FILE}\" \
    --parallax \
    --book-title     \"{BOOK_TITLE}\" \
    --character-name \"{CHARACTER_NAME}\" \
    --anthropic-key  \"$ANTHROPIC_API_KEY\" \
    --pexels-key     \"$PEXELS_API_KEY\" \
    --review-dir     \"{REVIEW_DIR}\"
"""

# Execute the command using get_ipython().system() for robustness
get_ipython().system(command)

print("🧩 Prebuild completed!")
print()
print("🎪 Expected log lines (confirm above):")
print("   Portrait pool detected at /content/resources/character — skipping pinned-portrait copy")
print("   Book cover directory pool detected at /content/resources/book_cover — skipping prebuild copy")
print("   Photo bank auto-detected at /content/resources/photo_bank (Path C) — if present:")
print("     photo_bank: N/59 image shots assigned from M bank photos")
print("     (curated photos become each assigned shot's chosen winner;")
print("      add --photo-bank-only to skip the web waterfall for those shots)")

In [ ]:
# Zip prebuild assets
import os
import zipfile

output_dir = REVIEW_DIR
zip_filename = REVIEW_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the "+REVIEW_DIR+" directory!")

In [ ]:
# Condition assets
!python condition_assets.py --review-dir {REVIEW_DIR} # --sr realesrgan]; --dry-run

In [ ]:
# OPTIONAL
# Zip conditioned assets
import os
import zipfile

output_dir = REVIEW_DIR
zip_filename = CONDITIONED_ZIP_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the", REVIEW_DIR, "directory!")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Render
# ════════════════════════════════════════════════════════════════════
import os
from pathlib import Path

Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Backgrounded so the next cell can tail LOG_FILE. Phase 3's render
# typically takes 10-15 min at 1080p; tailing is easier than waiting on
# a frozen cell.
!python render_plan.py \
    --plan              {PLAN_FILE} \
    --audio             {AUDIO_FILE} \
    --music             {MUSIC_BED} \
    --music-gain        {str(MUSIC_DB)} \
    --review-dir        {REVIEW_DIR} \
    --book-cover-pick   {str(BOOK_COVER_PICK)} \
    --book-cover-fit    {BOOK_COVER_FIT} \
    --book-cover-align  {BOOK_COVER_ALIGN} \
    --typography-family {TYPOGRAPHY_FAMILY} \
    --parallax \
    --typography-over-image \
    --no-captions \
    --grade             {GRADE} \
    --caption-backplate {CAPTION_BACKPLATE} \
    --output            {OUTPUT_FILE} \
    > {LOG_FILE} 2>&1 &

print("Rendering begins ...")
print(f"  log:    {LOG_FILE}")
print(f"  output: {OUTPUT_FILE}")
print()
print("Run the next cell to tail the log until completion.")
print("Rendering could nearly take up to 40 minutes!")


In [ ]:
# Monitor rendering progress
import time
from IPython.display import clear_output

log_path = LOG_FILE

print("Monitoring rendering progress...")
while True:
    try:
        # Read the log file contents
        try:
            with open(log_path, "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = ""

        # Clear cell output and show the last 20 lines
        clear_output(wait=True)
        lines = log_content.splitlines()
        print("\n".join(lines[-20:]))

        # Check if the script's success signature is in the log
        if "Done in" in log_content or "Rendered video →" in log_content:
            print("\n✅ Rendering process completed successfully! Stopped monitoring.")
            break

        time.sleep(5)

    except KeyboardInterrupt:
        print("\n⚠️ Monitoring stopped manually. The script may still be running.")
        break


In [ ]:
# !python diagnose_grade.py

In [ ]:
# !python diagnose_captions.py --plan output/re_generated_plan.json   # inspect actual ASS events

In [ ]:
# Zip output files for exporting
import os
import zipfile

output_dir = OUTPUT_BASE_DIR
zip_filename = FINAL_ZIP_FILE

# Get all files in the output directory
files_to_zip = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]

# Filter out the .mp3 file
filtered_files = [f for f in files_to_zip if not f.endswith('.mp3')]

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in filtered_files:
        # Add file to zip, preserving directory structure relative to 'output_dir'
        zipf.write(file_path, os.path.relpath(file_path, output_dir))

print(f"🤐 Successfully created '{zip_filename}' containing: ")
for f in filtered_files:
    print(f"  - {f}")

In [ ]:
# Save zipped file to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Define the destination directory and file path
dest_dir = DRIVE_SAVE_DIR
dest_file_path = os.path.join(dest_dir, FINAL_ZIP_FILE)

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)
print(f"Destination directory '{dest_dir}' ensured to exist.")

# --- Test write access to the directory ---
test_file = os.path.join(dest_dir, 'test_write.txt')
try:
    with open(test_file, 'w') as f:
        f.write('This is a test file.\n')
    print(f"✅ Successfully wrote test file to '{test_file}'.")
    os.remove(test_file) # Clean up the test file
    print(f"Test file '{test_file}' removed.")
except Exception as e:
    print(f"Error writing test file to '{test_file}': {e}")
    print("⚠️ It seems there might be a permissions or access issue with Google Drive.")
    # Exit or raise an error if write access fails
    raise
# ----------------------------------------

shutil.copy('/content/output_files.zip', dest_file_path)
print(f"📽️ Files are saved to Google Drive at '{dest_file_path}'.")